In [8]:
from langchain_core.messages import HumanMessage

In [9]:
import os
import json
from pathlib import Path
from tavily import TavilyClient
from dotenv import load_dotenv

load_dotenv()

def load_tavily_api_key():

    # Fall back to environment variable
    api_key = os.getenv("TAVILY_API_KEY")

    return api_key

# Initialize Tavily client
try:
    tavily_client = TavilyClient(api_key=load_tavily_api_key())
except Exception as e:
    print(f"Warning: Could not initialize Tavily client: {e}")
    tavily_client = None

def search_and_extract_info_by_link(url: str, extract_depth: str = "basic", include_images: bool = False):
    """
    Search and extract information from a URL using Tavily.
    
    Args:
        url (str): The URL of the web page to extract content from
        extract_depth (str): Extraction depth - "basic" or "advanced" (default: "basic")
        include_images (bool): Whether to include images in extraction (default: False)
    
    Returns:
        dict: Dictionary containing extracted content and metadata, or None if extraction fails
    """
    if tavily_client is None:
        print("Error: Tavily client not initialized. Please set TAVILY_API_KEY environment variable or configure it in config.json")
        return None
    
    try:
        # Extract content from the URL
        response = tavily_client.extract(
            urls=[url],
            include_images=include_images,
            extract_depth=extract_depth
        )
        
        # Check if extraction was successful
        if response.get("results") and len(response["results"]) > 0:
            result = response["results"][0]
            
            # Format the result
            extracted_info = {
                "url": result.get("url", url),
                "title": result.get("title", "No title"),
                "content": result.get("content", ""),
                "raw_content": result.get("raw_content", ""),
                "published_date": result.get("published_date"),
                "author": result.get("author"),
                "images": result.get("images", []) if include_images else []
            }
            
            return extracted_info
        else:
            print(f"Warning: No content extracted from URL: {url}")
            return None
            
    except Exception as e:
        print(f"Error extracting content from URL {url}: {str(e)}")
        return None



In [10]:
url = "https://docs.tavily.com/documentation/mcp"
info = search_and_extract_info_by_link(url)
info


{'url': 'https://docs.tavily.com/documentation/mcp',
 'title': 'Tavily MCP Server - Tavily Docs',
 'content': '',
 'raw_content': '[Tavily Docs home page](https://tavily.com/)\n\n[Home](/welcome)[Documentation](/documentation/about)[SDKs](/sdk/python/quick-start)[Examples](/examples/use-cases/chat)[FAQ](/faq/faq)[Changelog](/changelog/changelog)\n\n* [API Playground](https://app.tavily.com/playground)\n* [Community](https://community.tavily.com)\n* [Blog](https://blog.tavily.com)\n\n* [Credits & Pricing](/documentation/api-credits)\n* [Rate Limits](/documentation/rate-limits)\n\n##### API Reference\n\n* [Introduction](/documentation/api-reference/introduction)\n* [POST\n\n  Tavily Search](/documentation/api-reference/endpoint/search)\n* [POST\n\n  Tavily Extract](/documentation/api-reference/endpoint/extract)\n* [POST\n\n  Tavily Crawl](/documentation/api-reference/endpoint/crawl)\n* [POST\n\n  Tavily Map](/documentation/api-reference/endpoint/map)\n* [GET\n\n  Usage](/documentation/ap